# Mounting the Google Drive

In [ ]:
import random
import numpy as np
import torch

seed = 42

# Python
random.seed(seed)

# NumPy
np.random.seed(seed)

# PyTorch
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

# CUDA determinism
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# Importing needed Libraries

In [ ]:
import unicodedata
import re
from tqdm import tqdm

In [ ]:
with open("/content/drive/MyDrive/writingPrompts/train.wp_source") as f:
    source_lines = f.readlines()

with open("/content/drive/MyDrive/writingPrompts/train.wp_target") as f:
    target_lines = f.readlines()

print(len(source_lines), len(target_lines))

272600 272600


## Initial Exploration and Cleaning of the dataset

In [ ]:
dataset = []

for i in range(len(source_lines)):
    prompt = source_lines[i].strip()
    story = target_lines[i].strip()

    dataset.append({
        "id": f"wp_{i}",
        "prompt": prompt,
        "story": story
    })


In [ ]:
import random

SEED = 42
random.seed(SEED)

random.shuffle(dataset)

In [ ]:
import re
import unicodedata


def normalize_unicode(text):
    return unicodedata.normalize("NFKC", text)


def clean_whitespace(text):
    text = normalize_unicode(text)
    text = text.replace("<newline>", "\n")
    text = text.replace("\r\n", "\n")
    text = text.replace("\r", "\n")

    # Keep paragraph breaks, but clean spaces inside lines
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)

    return text.strip()


def has_bad_artifacts(story):
    lower = story.lower()

    bad_patterns = [
        "[wp]",
        "[rf]",
        "[eu]",
        "this is my first",
        "this is only my",
        "please point out",
        "criticism",
        "i need you to write",
        "according to all instructions",
        "###",
        "***",
    ]

    return any(p in lower for p in bad_patterns)


def has_clean_final_sentence(story):
    story = story.strip()

    if not story:
        return False

    if story[-1] not in [".", "!", "?", '"', "”"]:
        return False

    sentences = re.split(r"(?<=[.!?])\s+", story)
    last_sentence = sentences[-1].strip()

    if len(last_sentence.split()) < 4:
        return False

    bad_last_words = {
        "and", "but", "or", "the", "a", "an",
        "to", "of", "as", "with", "in", "from"
    }

    last_word = last_sentence.lower().split()[-1].strip(".,!?\"”'")

    return last_word not in bad_last_words


def has_repetition_problem(story):
    sentences = re.split(r"(?<=[.!?])\s+", story.strip())
    normalized = [
        s.lower().strip()
        for s in sentences
        if len(s.strip()) > 20
    ]

    return len(normalized) != len(set(normalized))


def is_valid_story(story):
    if not story:
        return False

    story_lower = story.lower().strip()

    if story_lower in ["[deleted]", "[removed]", "deleted", "removed", ""]:
        return False

    if has_bad_artifacts(story):
        return False

    word_count = len(story.split())

    if word_count < 180:
        return False

    if word_count > 450:
        return False

    if not has_clean_final_sentence(story):
        return False

    if has_repetition_problem(story):
        return False

    return True

In [ ]:
# Build clean_rows from dataset before weak genre splitting

clean_rows = []

for row in dataset:
    prompt = clean_whitespace(row["prompt"])
    story = clean_whitespace(row["story"])

    if is_valid_story(story):
        clean_rows.append({
            "id": row["id"],
            "prompt": prompt,
            "story": story
        })

print("Raw dataset size:", len(dataset))
print("Clean rows:", len(clean_rows))

Raw dataset size: 272600
Clean rows: 54900


In [ ]:
def weak_genre_scores(prompt, story):
    text = (prompt + " " + story).lower()

    fantasy_keywords = [
        "magic", "wizard", "witch", "dragon", "kingdom", "curse", "spell",
        "sword", "elf", "dwarf", "goblin", "demon", "angel", "heaven",
        "hell", "god", "goddess", "genie", "prophecy", "fairy", "monster",
        "vampire", "werewolf", "sorcerer", "necromancer", "portal",
        "underworld", "afterlife", "ghost", "spirit"
    ]

    romance_keywords = [
        "love", "loved", "lover", "romance", "romantic", "kiss", "crush",
        "date", "dating", "boyfriend", "girlfriend", "husband", "wife",
        "fiance", "fiancée", "wedding", "marriage", "married", "heart",
        "relationship", "breakup", "ex", "soulmate", "valentine",
        "confession", "proposal", "jealous", "affection"
    ]

    scifi_keywords = [
        "space", "spaceship", "starship", "planet", "alien", "aliens",
        "galaxy", "universe", "robot", "android", "cyborg", "ai",
        "artificial intelligence", "time travel", "timeline", "future",
        "simulation", "virtual", "quantum", "laser", "meteor", "asteroid",
        "colony", "mars", "moon", "interstellar", "dimension",
        "technology", "machine", "mech", "clone", "terraform"
    ]

    scores = {
        "fantasy": sum(k in text for k in fantasy_keywords),
        "romance": sum(k in text for k in romance_keywords),
        "sci-fi": sum(k in text for k in scifi_keywords),
    }

    return scores

In [ ]:
fantasy_candidates = []
romance_candidates = []
scifi_candidates = []
mixed_or_unknown = []

MIN_SCORE = 1

for row in clean_rows:
    scores = weak_genre_scores(row["prompt"], row["story"])
    best_genre = max(scores, key=scores.get)
    best_score = scores[best_genre]

    row_copy = dict(row)
    row_copy["weak_scores"] = scores
    row_copy["weak_genre"] = best_genre

    if best_score < MIN_SCORE:
        mixed_or_unknown.append(row_copy)
    elif best_genre == "fantasy":
        fantasy_candidates.append(row_copy)
    elif best_genre == "romance":
        romance_candidates.append(row_copy)
    elif best_genre == "sci-fi":
        scifi_candidates.append(row_copy)

print("Fantasy candidates:", len(fantasy_candidates))
print("Romance candidates:", len(romance_candidates))
print("Sci-Fi candidates:", len(scifi_candidates))
print("Unknown/mixed:", len(mixed_or_unknown))

Fantasy candidates: 21635
Romance candidates: 19307
Sci-Fi candidates: 13578
Unknown/mixed: 380


In [ ]:
import json

def save_json(path, data):
    with open(path, "w") as f:
        json.dump(data, f, indent=2)

save_json("/content/drive/MyDrive/fantasy_candidates.json", fantasy_candidates)
save_json("/content/drive/MyDrive/romance_candidates.json", romance_candidates)
save_json("/content/drive/MyDrive/scifi_candidates.json", scifi_candidates)
save_json("/content/drive/MyDrive/mixed_or_unknown.json", mixed_or_unknown)

print("Weak candidate files saved ✅")

Weak candidate files saved ✅


In [ ]:
import json
import os

def manual_label_genre(
    candidate_samples,
    target_genre,
    save_file,
    state_file,
    target_count=500,
    save_every=5
):
    # Load labeled data
    if os.path.exists(save_file):
        with open(save_file, "r") as f:
            labeled = json.load(f)
        print(f"Loaded {len(labeled)} labeled {target_genre} samples ✅")
    else:
        labeled = []

    # Load state
    if os.path.exists(state_file):
        with open(state_file, "r") as f:
            state = json.load(f)
        start_idx = state["index"]
        accepted_count = state["accepted_count"]
        print(f"Resuming {target_genre} from index {start_idx} 🔄")
    else:
        start_idx = 0
        accepted_count = len(labeled)

    count = start_idx

    for i in range(start_idx, len(candidate_samples)):
        item = candidate_samples[i]

        if accepted_count >= target_count:
            print(f"Target reached for {target_genre} ✅")
            break

        print("\n" + "=" * 80)
        print(f"INDEX: {i}")
        print(f"TARGET GENRE: {target_genre}")
        print(f"Accepted: {accepted_count}/{target_count}")

        print("\nPROMPT:\n")
        print(item["prompt"])

        print("\nSTORY PREVIEW:\n")
        print(item["story"][:1200])

        print("\nWeak scores:", item.get("weak_scores", {}))

        label = input(
            f"\n{count} Keep as {target_genre}? "
            f"(y=keep, n/enter=skip, q=quit): "
        ).strip().lower()

        count += 1

        if label == "q":
            print("Saving and exiting safely... 💾")
            break

        if label == "y":
            clean_item = {
                "id": item["id"],
                "genre": target_genre,
                "story_idea": item["prompt"],
                "story": item["story"]
            }

            labeled.append(clean_item)
            accepted_count += 1

        # Auto save
        if count % save_every == 0:
            with open(save_file, "w") as f:
                json.dump(labeled, f, indent=2)

            with open(state_file, "w") as f:
                json.dump({
                    "index": i + 1,
                    "accepted_count": accepted_count
                }, f, indent=2)

            print("Progress saved ✅")

    # Final save
    with open(save_file, "w") as f:
        json.dump(labeled, f, indent=2)

    with open(state_file, "w") as f:
        json.dump({
            "index": i + 1,
            "accepted_count": accepted_count
        }, f, indent=2)

    print(f"Final save complete for {target_genre} 💾🔥")
    print(f"Total accepted: {accepted_count}")

    return labeled

In [ ]:
fantasy_labeled = manual_label_genre(
    candidate_samples=fantasy_candidates,
    target_genre="fantasy",
    save_file="/content/drive/MyDrive/fantasy_labeled.json",
    state_file="/content/drive/MyDrive/fantasy_progress.json",
    target_count=500,
    save_every=5
)

Loaded 351 labeled fantasy samples ✅
Resuming fantasy from index 1103 🔄

INDEX: 1103
TARGET GENRE: fantasy
Accepted: 351/500

PROMPT:

[ WP ] A cop arrives at the golden gate bridge to talk a man out of committing suicide . After they have a short conversation , the cop jumps off the bridge .

STORY PREVIEW:

San Francisco was bathed in grey fog as the sun began to rise on the horizon . The Golden Gate Bridge was barely visible in the hazy morning sky as Officer Dalton made his way to the bridge with his sirens blasting . He had received a report of a civilian standing on the railing preparing to jump . This wasn ’ t uncommon for the Golden Gate Bridge . 
 
 Officer Dalton drove slowly across bridge when he spotted a dark figure along the railing . He parked the police car and slowly made his way towards the shadowy outline of a person . It was almost impossible to see through the thick fog , but he could tell that the person was standing facing the water with their arms outstretched b

In [ ]:
romance_labeled = manual_label_genre(
    candidate_samples=romance_candidates,
    target_genre="romance",
    save_file="/content/drive/MyDrive/romance_labeled.json",
    state_file="/content/drive/MyDrive/romance_progress.json",
    target_count=500,
    save_every=5
)

Streaming output truncated to the last 5000 lines.

“ The usual Pumpkin ? ” , a tall glass of orange juice is set down before me . I felt like the waitress may be playing with me at this point . It 's tough to remember . Yesterday ... I 'm certain it was hun , and the day before , what was the day before ? “ Please ” I stammer , peering up at Jackie . I suppose her name is Jackie , I have few paths to truth anymore , but for what my eyes provide . That may be why I spend so much time in businesses which require the use of name tags . “ You know , I 've always found it so strange that some people eat maple syrup on their eggs. ” The world gives me these details , and I must immediately squirrel them away , to be hidden for a time when my eyes again fail to connect to what I 've seen . 
 
 
 “ Thank you , Jackie ” . “ Excuse me , sir ? ” a gruff voice issues from behind my left ear , my perspective jarringly shifts by ninety degrees , I could swear I was sitting in the booth opposite of 

In [ ]:
scifi_labeled = manual_label_genre(
    candidate_samples=scifi_candidates,
    target_genre="sci-fi",
    save_file="/content/drive/MyDrive/scifi_labeled.json",
    state_file="/content/drive/MyDrive/scifi_progress.json",
    target_count=500,
    save_every=5
)

Loaded 351 labeled sci-fi samples ✅
Resuming sci-fi from index 1199 🔄

INDEX: 1199
TARGET GENRE: sci-fi
Accepted: 351/500

PROMPT:

[ WP ] A flyer saucer lands on earth . A nervous crowd gathers as the doors open . Out walks a normal , boring guy named Kevin .

STORY PREVIEW:

“ This is a historical moment for the whole mankind . The first time that we have close encounter . As you can see behind me on the lawn of the White House the alien spaceship just landed . We are all really excited here. ” - said the reporter in front of one of the many TV-news cameras around the fence . 
 
 The secret service is well prepared for everything . As well as the military whose personnel constantly monitoring every move of the saucer . They are ready to destroy it for the first suspicious move . 
 
 “ I can see it , the door is opening up now ! ” - shouts the reporter . Then everybody got quiet . This is not what they were expecting . They thought there will be some strange looking creature from an o

In [ ]:
with open("/content/drive/MyDrive/fantasy_labeled.json", "r") as f:
    fantasy_labeled = json.load(f)

with open("/content/drive/MyDrive/romance_labeled.json", "r") as f:
    romance_labeled = json.load(f)

with open("/content/drive/MyDrive/scifi_labeled.json", "r") as f:
    scifi_labeled = json.load(f)

final_rows = fantasy_labeled + romance_labeled + scifi_labeled

print("Final dataset size:", len(final_rows))

Final dataset size: 1020


In [ ]:
from collections import Counter

print(Counter(row["genre"] for row in final_rows))

Counter({'sci-fi': 365, 'fantasy': 353, 'romance': 302})


In [ ]:
with open("/content/drive/MyDrive/final_genre_story_dataset.json", "w") as f:
    json.dump(final_rows, f, indent=2)

print("Final combined dataset saved ✅")

Final combined dataset saved ✅


In [ ]:
import json
import os

input_file = "/content/drive/MyDrive/final_genre_story_dataset.json"

# Create folder name
output_dir = "data_preprop"
os.makedirs(output_dir, exist_ok=True)  # creates folder if not exists

# Define output file inside folder
output_file = os.path.join(output_dir, "processed_stories.jsonl")

# Read JSONL file and write to output
with open(input_file, "r") as infile, open(output_file, "w") as outfile:
    for line in infile:
        item = json.loads(line)

        # Extract fields
        prompt = item["story_idea"]
        story = item["story"]
        genre = item["genre"]

        # Clean text
        prompt = prompt.replace("<newline>", "\n").strip()
        story = story.replace("<newline>", "\n").strip()

        # Format text
        formatted_text = f"""<GENRE> {genre}

<PROMPT>
{prompt}

<STORY>
{story}
"""

        # Save in training-ready format
        new_item = {"text": formatted_text}

        outfile.write(json.dumps(new_item) + "\n")

print(f"File saved at: {output_file}")

File saved at: data_preprop/processed_stories.jsonl


In [ ]:
import json

with open("/content/data_preprop/processed_stories.jsonl") as f:
    sample = json.loads(f.readline())

print(sample.keys())
print(sample["text"][:500])

dict_keys(['text'])
<GENRE> fantasy

<PROMPT>
[ WP ] Over night , 90 % of the world 's population has dropped dead . In the following weeks , the survivors , who come from diverse countries , ethnicities , religious beliefs and lifestyles realize that they all share a single , peculiar trait ...

<STORY>
The world came crashing down in minutes . Many of us were asleep when it happened , and did n't find out about it until later . When we awoke , we saw the carnage spread through the land , and we wept . 
 
 There w
